# Pick a confidence threshold

`0.65` is this platform's default, not your answer. The threshold decides what
counts as a detection, and the right value depends on which mistake costs more —
a missed lesion or a false one. This measures both across the range so the
choice is made from a curve instead of from the default.

In [ ]:
import orinth

MODEL_ID = next(
    model.id
    for model in orinth.models.list()
    if model.available and model.task_type in {"object_detection", "segmentation"}
)
DATASET_ID = next(
    dataset.id
    for dataset in orinth.datasets.list()
    if dataset.task_type in {"object_detection", "segmentation"} and dataset.splits.get("valid")
)
print(MODEL_ID, "on", DATASET_ID)

predictor = orinth.models.predictor(MODEL_ID)
frame = orinth.datasets.load(DATASET_ID, "valid", limit=30)
print(frame.height, "items")

## Predict once, threshold many times

The expensive part is the forward pass, and it does not depend on the threshold.
Predicting at the lowest value in the sweep and filtering afterwards is the same
answer for a fraction of the time — the mistake worth avoiding is re-running the
model once per threshold.

In [ ]:
from pathlib import Path

FLOOR = 0.05
params = orinth.models.parameters(confidence_threshold=FLOOR)

items = []
for row in frame.iter_rows(named=True):
    detections = predictor.predict(Path(row["path"]), params)
    truth = {annotation["label"] for annotation in (row["annotations"] or [])}
    items.append(
        {
            "truth": truth,
            "predicted": [(d.class_name, float(d.confidence)) for d in detections],
        }
    )

print(sum(len(item["predicted"]) for item in items), "detections above", FLOOR)

## The sweep

Counted per image, not per box: "did this image get flagged for the right class"
is the question the studio's image-level verdict answers, and matching the metric
to the decision is the point.

In [ ]:
import polars as pl

rows = []
for threshold in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.8, 0.9]:
    hits = misses = false_alarms = 0
    for item in items:
        kept = {name for name, score in item["predicted"] if score >= threshold}
        hits += len(kept & item["truth"])
        misses += len(item["truth"] - kept)
        false_alarms += len(kept - item["truth"])
    precision = hits / (hits + false_alarms) if hits + false_alarms else 0.0
    recall = hits / (hits + misses) if hits + misses else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    rows.append(
        {"threshold": threshold, "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3)}
    )

sweep = pl.DataFrame(rows)
print(sweep)
best = sweep.sort("f1", descending=True).row(0, named=True)
print("\nbest F1 at threshold", best["threshold"], "->", best["f1"])

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(6.5, 3.2))
axis.plot(sweep["threshold"], sweep["precision"], marker="o", label="precision")
axis.plot(sweep["threshold"], sweep["recall"], marker="o", label="recall")
axis.plot(sweep["threshold"], sweep["f1"], marker="o", label="F1")
axis.axvline(0.65, linestyle="--", linewidth=1, label="platform default")
axis.set_xlabel("confidence threshold")
axis.legend()
figure.tight_layout()
plt.show()

F1 picks the balanced point, which is the right answer only when the two
mistakes cost the same. They usually do not — read recall at the precision you
can live with, and set the threshold there.

In [ ]:
with orinth.runs.start(f"threshold sweep {MODEL_ID}", dataset_id=DATASET_ID) as run:
    for row in sweep.iter_rows(named=True):
        run.log(precision=row["precision"], recall=row["recall"], f1=row["f1"])
    run.log_text("chosen", f"threshold={best['threshold']} f1={best['f1']}")
    run.log_artifact(sweep.write_csv().encode("utf-8"), name="threshold-sweep.csv")